# Load Pretrained Model

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

# Test BEFORE Fine-Tuning

In [2]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate("Explain machine learning in simple terms"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Explain machine learning in simple terms.












































# Create Instruction Dataset

In [3]:
dataset = [
    {"input": "Explain machine learning simply",
     "output": "Machine learning is when computers learn patterns from data to make decisions."},

    {"input": "List 3 benefits of exercise",
     "output": "1. Improves health\n2. Reduces stress\n3. Increases energy"},

    {"input": "What is Python?",
     "output": "Python is a programming language used for building applications and analyzing data."}
]

# Format Data for Training

In [4]:
def format_example(example):
    return f"Instruction: {example['input']}\nResponse: {example['output']}"

train_texts = [format_example(x) for x in dataset]

# Fine-Tune (SFT)

In [5]:
from transformers import Trainer, TrainingArguments
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts})

def tokenize(example):
    tokenized = tokenizer(example["text"], truncation=True, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized = train_dataset.map(tokenize)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

trainer.train()

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,10.690346
2,8.279552
3,4.004951
4,2.072260
5,1.597429
6,0.713191


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=4.55962148308754, metrics={'train_runtime': 46.9956, 'train_samples_per_second': 0.192, 'train_steps_per_second': 0.128, 'total_flos': 2351670755328.0, 'train_loss': 4.55962148308754, 'epoch': 3.0})

In [6]:
print(generate("Explain machine learning simply"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Explain machine learning simply means that you can learn a lot of things from it.
